# Model validation

In [1]:
from model import STGAETransformerPoolingMultiGCN
import torch 
import xarray as xr

## Load the model

In [ ]:
hidden_channels = 64
latent_dim = 16
num_epochs = 8
initial_lr = 0.001    
lr_decay_epochs = num_epochs-2 
lr_gamma = 0.5       
checkpoint_dir = "./anomaly_detector"
accumulation_steps = 2
temp_window = 25
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = STGAETransformerPoolingMultiGCN(temp_window*2, hidden_channels, latent_dim).to(device)
model.load_state_dict(torch.load('anomaly_detector/stgae_epoch_8.pth', weights_only=True))

<All keys matched successfully>

## Load the dataset

In [3]:
import os 
lat_min, lat_max = 30, 60
lon_min, lon_max = 0, 30
time_window = 20

dataset_base_directory = os.curdir+os.sep+"dataset"
dataset_files = dataset_base_directory+os.sep+'*.nc4'

ds = xr.open_mfdataset(dataset_files, cache=False)
ds

<xarray.Dataset> Size: 745GB
Dimensions:  (time: 5708, lat: 3298, lon: 9896)
Coordinates:
  * lon      (lon) float32 40kB -180.0 -179.9 -179.9 ... 179.9 179.9 180.0
  * lat      (lat) float32 13kB -59.98 -59.95 -59.91 ... 59.91 59.95 59.98
  * time     (time) datetime64[ns] 46kB 2020-01-01 ... 2020-04-28T21:30:00.00...
Data variables:
    Tb       (time, lat, lon) float32 745GB dask.array<chunksize=(1, 1649, 2474), meta=np.ndarray>
Attributes:
    BeginDate:       2020-01-01
    BeginTime:       00:00:00.000Z
    EndDate:         2020-01-01
    EndTime:         00:59:59.999Z
    FileHeader:      StartGranuleDateTime=2020-01-01T00:00:00.000Z;\nStopGran...
    InputPointer:    merg_2020010100_4km-pixel
    title:           NCEP/CPC 4km Global (60N - 60S) IR Dataset
    ProductionTime:  2020-01-03T00:19:23.203Z

In [4]:
ds = ds.sel(
    lat=slice(lat_min, lat_max), 
    lon=slice(lon_min, lon_max) 
)
ds = ds.chunk({'time': time_window})
ds['Tb'] = ds['Tb'].astype('float16')

Select a subset of the dataset for the network performance testing

In [5]:
import random
t1 = random.randint(0, ds['time'].shape[0]-temp_window)
ds_time_slice = ds.isel(time=slice(t1, t1+temp_window))

## Prepare poisoned data

In [6]:
from attacks import *
from torch import autocast
import torch.nn.functional as F

In [7]:
def compute_loss(x_recon, target, loss_type="mse", temp_weight=1.0, mask_weight=0.2):
    T = x_recon.size(1) // 2
    recon_temp = x_recon[:, :T]
    target_temp = target[:, :T]
    recon_mask = x_recon[:, T:]
    target_mask = target[:, T:]
    if loss_type == "mse":
        loss_temp = F.mse_loss(recon_temp, target_temp, reduction='mean')
        loss_mask = F.mse_loss(recon_mask, target_mask, reduction='mean')
    elif loss_type == "mae":
        loss_temp = F.l1_loss(recon_temp, target_temp, reduction='mean')
        loss_mask = F.l1_loss(recon_mask, target_mask, reduction='mean')
    elif loss_type == "smooth_l1":
        loss_temp = F.smooth_l1_loss(recon_temp, target_temp, reduction='mean')
        loss_mask = F.smooth_l1_loss(recon_mask, target_mask, reduction='mean')
    else:
        raise ValueError("Unsupported loss type: " + loss_type)

    return temp_weight * loss_temp + mask_weight * loss_mask
loss_type = 'mse'
def evaluate_sample(data, model, device):
    model.eval()
    data = data.to(device)
    with torch.no_grad():
        with autocast('cuda'):
            x_recon, z, perm = model(data)
    error = compute_loss(x_recon, data.x[perm], loss_type=loss_type)
    del data, x_recon
    torch.cuda.empty_cache()
    return error, z

In [8]:

poisoned_samples = {
    'Fixed Bias': [
        {
            'desc':f'fixed bias of {bias}',
            'attack': fixed_bias(ds_time_slice, bias) 
        } for bias in range(40, 50, 2)
    ],
    'Incrementing Bias': [
        {
            'desc':f'incrementing bias of {bias_rate} every half an hour',
            'attack':incrementing_bias(ds_time_slice, bias_rate) 
        } for bias_rate in range(2, 12, 2)
    ],
    'Adversarial Perturbation': [
        {
            'desc': f'Adversarial Perturbation of {epsilon}',
            'attack': adversarial_perturbation(ds_time_slice, epsilon)
        }
        for epsilon in range(5, 20, 2)
    ],
    'Sensor spoofing': [
        {
            'desc': f'Random sensor spoofing',
            'attack': rnd_sensor_spoofing(ds_time_slice, 50, 100, 50, 100)
        }
         for i in range(5)
    ],
    'Data block jamming': [
        {
            'desc': f'Data block jamming with {fraction*10}%',
            'attack': data_block_jamming(ds_time_slice, fraction/10, main_var='Tb') 
        }
        for fraction in range(1, 10)
    ],
    'Temporal shifting': [
        {
            'desc': f'Temporal shifting of {shift}',
            'attack': temporal_shifting(ds_time_slice,shift, main_var='Tb') 
        }
        for shift in range(0, 10, 2)
    ],
    'Spatial shifting': [
        {
            'desc': f'Random spatial shifting',
            'attack': rnd_spatial_shifting(ds_time_slice,200)
        }
         for i in range(5)
    ],
    'Miscalibration error': [
        {
            'desc': f'Miscalibration error of {err/10}',
            'attack': miscalibration_error_injection(ds_time_slice, err/10)
        }
        for err in range(10, 100,10)
    ],
    'Replay attack': [
        {
            'desc': f'Replay attack of period={period}',
            'attack': replay_attack(ds_time_slice, period) 
        }
        for period in range(1, 3)
    ],
    'Backdoor attack': [
        {
            'desc': f'Backdoor attack of strength={strength}',
            'attack': backdoor_trigger(ds_time_slice, strength) 
        }
         for strength in range(1, 10)
    ]
}

In [9]:
from dataset import chunk_to_graph

In [10]:
errors = []
mean_temp = 272.51712
std_temp = 15.239919

for label, attacks in poisoned_samples.items():
    print(f'''
          #########  {label}  #########
          ''')
    for attack in attacks:
        attack_norm = (attack['attack']-mean_temp)/std_temp
        graphed_sample = chunk_to_graph(attack_norm['Tb'], temp_window)
        error, latent = evaluate_sample(graphed_sample, model, device)
        err = error.mean().item()
        errors.append(err)
        print(f"{attack['desc']} - average reconstruction error per node:", error.mean().item())


          #########  Fixed Bias  #########
          
fixed bias of 40 - average reconstruction error per node: 0.23249457776546478
fixed bias of 42 - average reconstruction error per node: 0.2522971034049988
fixed bias of 44 - average reconstruction error per node: 0.2697262763977051
fixed bias of 46 - average reconstruction error per node: 0.29791486263275146
fixed bias of 48 - average reconstruction error per node: 0.3135160207748413

          #########  Incrementing Bias  #########
          
incrementing bias of 2 every half an hour - average reconstruction error per node: 0.33881065249443054
incrementing bias of 4 every half an hour - average reconstruction error per node: 1.1415929794311523
incrementing bias of 6 every half an hour - average reconstruction error per node: 2.530498504638672
incrementing bias of 8 every half an hour - average reconstruction error per node: 4.428591728210449
incrementing bias of 10 every half an hour - average reconstruction error per node: 6.898

In [18]:
anomaly_threshold = 0.27
precision = 0
for error in errors:
    precision += 1 if error>anomaly_threshold else 0
print(f"Anomalies were detected with {precision/len(errors)*100}% accuracy")

Anomalies were detected with 66.12903225806451% accuracy
